## Setting up InfluxDB 

The first thing you'll need to do is ensure you have Python 3 installed, this will not work with older versions of Python.
Install the influxdb3-python module. You'll use this to write and query InfluxDB. Run the command below in your terminal:

In [97]:
!pip install influxdb3-python


Lastly, let's install pandas which will be helpful for organizing our data output when we query.

In [98]:
import pandas as pd
import numpy as np

Get the permission to access the database

Here, we initialize the token, organization info, and server url host that are needed to set up the initial connection to InfluxDB. The client connection is then established with the InfluxDBClient initialization.

In [99]:
import os, time
from influxdb_client_3 import InfluxDBClient3, Point

#add this to load and upadte the token file
%reload_ext dotenv
%dotenv

token = os.environ.get("INFLUXDB_TOKEN")
org = "Air quality - Harrisburg and Philadelphia"
host = "https://us-east-1-1.aws.cloud2.influxdata.com"

client = InfluxDBClient3(host=host, token=token, org=org)

## Example of a dataset and the querrying process

Uploading the data

In [100]:
database="pilot-harrisburg-home"

data = {
  "point1": {
    "location": "Klamath",
    "species": "bees",
    "count": 23,
  },
  "point2": {
    "location": "Portland",
    "species": "ants",
    "count": 30,
  },
  "point3": {
    "location": "Klamath",
    "species": "bees",
    "count": 28,
  },
  "point4": {
    "location": "Portland",
    "species": "ants",
    "count": 32,
  },
  "point5": {
    "location": "Klamath",
    "species": "bees",
    "count": 29,
  },
  "point6": {
    "location": "Portland",
    "species": "ants",
    "count": 40,
  },
}

for key in data:
  point = (
    Point("census")
    .tag("location", data[key]["location"])
    .field(data[key]["species"], data[key]["count"])
  )
  client.write(database=database, record=point)
  time.sleep(1) # separate points by 1 second

print("Complete. Return to the InfluxDB UI.")


Complete. Return to the InfluxDB UI.


### A sample querry in python 

In [101]:
query = """SELECT *
FROM 'census'
WHERE time >= now() - interval '24 hours'
AND ('bees' IS NOT NULL OR 'ants' IS NOT NULL)"""

# Execute the query
table = client.query(query=query, database="pilot-harrisburg-home", language='sql') 

# Convert to dataframe
df = table.to_pandas().sort_values(by="time")
df = pd.DataFrame(df)
df


,ants,bees,location,time
6,NaN,23.0,Klamath,2025-12-30 18:16:36.910444392
12,30.0,NaN,Portland,2025-12-30 18:16:37.957687959
7,NaN,28.0,Klamath,2025-12-30 18:16:39.004579895
13,32.0,NaN,Portland,2025-12-30 18:16:40.128996893
8,NaN,29.0,Klamath,2025-12-30 18:16:41.213575379
14,40.0,NaN,Portland,2025-12-30 18:16:42.259226257
9,NaN,23.0,Klamath,2025-12-30 19:59:41.347587416
15,30.0,NaN,Portland,2025-12-30 19:59:42.390911325
10,NaN,28.0,Klamath,2025-12-30 19:59:43.432818726
16,32.0,NaN,Portland,2025-12-30 19:59:44.475855861


Aggregate functions take the values of all rows in a table and use them to perform an aggregate operation. The result is output as a new value in a single-row table.

In this example, we use the mean() function to calculate the average value of data points in the last 10 minutes.

In [102]:
query = """SELECT mean(count)
FROM "census"
WHERE time > now() - 10m"""

# Execute the query
table = client.query(query=query, database="pilot-harrisburg-home", language="influxql")

# Convert to dataframe
df = table.to_pandas().sort_values(by="time")
print(df)


KeyError: 'time'

## Connecting Pilot Test Devices to the database

#### The quantaq devices


In [ ]:
#Load the api key from the envrionment variable 
%reload_ext dotenv
%dotenv
QUANTAQ_APIKEY = os.environ.get("QUANTAQ_APIKEY")

First attempt to print the account information as a test

In [ ]:
import requests 
import json

#Define the the account url as defined in the quantaq documentation 
qauntaq_aq_account_url = "https://api.quant-aq.com/v1/account/"

#Use the requests module to obtain the account information
response = requests.get(qauntaq_aq_account_url, auth=(QUANTAQ_APIKEY,""))

#Print the responses for verification
response_json_acc = response.content
response_dict = json.loads(response_json_acc)

#Print the response veritcally 
for info in response_dict.items():
    print(info)




('confirmed', True)
('email', 'izabayoalain@gmail.com')
('first_name', 'Alain')
('id', 3175)
('is_administrator', False)
('last_name', 'Izabayo')
('last_seen', '2025-12-30T18:35:59.856359')
('member_since', '2025-10-22T02:02:36.104424')
('role', 1)
('username', 'Izaalain')


#### Automation 
First create a function that automatates api calls from quantaq

In [ ]:
def get_quantaq_data(url_ending):
    """
    A function that takes a list of words that go immediately after
    after the "https://api.quant-aq.com" of the url and help to 
    get access to scrap data from quantaq

    Args: 
        url_list (list):    A list of strings to be added to the url 
    
    Returns:
        response_json (json): a json string with the requested data        
    """

    #Define the fist part of the url
    url_first_part = "https://api.quant-aq.com/"

    #Initialize a continieoulsy increasing string to be used to complete the url
    url_additional = ""

    #Complete the url with user input strings 
    for adds_on in url_ending:
        url_additional = url_additional+adds_on+"/"        
    complete_url = url_first_part+url_additional

    #Make the call of the response to be returned 
    response_json = requests.get(complete_url, auth=(QUANTAQ_APIKEY,""))

    return response_json.content

In [ ]:
get_quantaq_data(['v1', 'devices', 'MOD-X-00959','data'])

b'{"data":[{"co":199.9,"geo":{"lat":40.3156,"lon":-76.8962},"met":{"wx_u":0.0,"wx_v":0.0,"wx_wd":0.0,"wx_ws":0.0,"wx_ws_scalar":0.0},"model":{"gas":{"co":20508,"no":20509,"no2":20510,"o3":20511},"pm":{"pm1":18867,"pm10":18869,"pm25":18868}},"no":-0.43,"no2":6.99,"o3":37.3,"pm1":0.04,"pm10":0.22,"pm25":0.04,"raw_data_id":19965589,"sn":"MOD-X-00959","timestamp":"2025-12-30T23:19:58","timestamp_local":"2025-12-30T18:19:58","url":"https://api.quant-aq.com/v1/devices/MOD-X-00959/data/19965537"},{"co":197.72,"geo":{"lat":40.3156,"lon":-76.8962},"met":{"wx_u":null,"wx_v":0.0,"wx_wd":0.0,"wx_ws":0.0,"wx_ws_scalar":0.0},"model":{"gas":{"co":20508,"no":20509,"no2":20510,"o3":20511},"pm":{"pm1":18867,"pm10":18869,"pm25":18868}},"no":1.46,"no2":6.44,"o3":37.12,"pm1":0.04,"pm10":8.24,"pm25":0.05,"raw_data_id":19965587,"sn":"MOD-X-00959","timestamp":"2025-12-30T23:18:58","timestamp_local":"2025-12-30T18:18:58","url":"https://api.quant-aq.com/v1/devices/MOD-X-00959/data/19965535"},{"co":203.01,"geo":

In [123]:

binary_data = get_quantaq_data(['v1', 'devices', 'MOD-X-00959', 'data', '?page=1917&per_page=50'])
json_data = binary_data.decode("utf-8") 
dict_data = json.loads(json_data)
dict_data['meta']
df_test = pd.DataFrame(dict_data['data'])
df_test

,co,geo,met,model,no,no2,o3,pm1,pm10,pm25,raw_data_id,sn,timestamp,timestamp_local,url
0,None,"{'lat': None, 'lon': None}","{'wx_u': 0.0, 'wx_v': 0.0, 'wx_wd': 0.0, 'wx_w...","{'gas': {'co': None, 'no': None, 'no2': None, ...",None,None,None,None,None,None,2781314,MOD-X-00959,2025-06-16T14:42:45,2025-06-16T14:42:45,https://api.quant-aq.com/v1/devices/MOD-X-0095...
1,None,"{'lat': None, 'lon': None}","{'wx_u': 0.0, 'wx_v': 0.0, 'wx_wd': 0.0, 'wx_w...","{'gas': {'co': None, 'no': None, 'no2': None, ...",None,None,None,None,None,None,2781312,MOD-X-00959,2025-06-16T14:41:45,2025-06-16T14:41:45,https://api.quant-aq.com/v1/devices/MOD-X-0095...
2,None,"{'lat': None, 'lon': None}","{'wx_u': 0.0, 'wx_v': 0.0, 'wx_wd': 0.0, 'wx_w...","{'gas': {'co': None, 'no': None, 'no2': None, ...",None,None,None,None,None,None,2781311,MOD-X-00959,2025-06-16T14:40:45,2025-06-16T14:40:45,https://api.quant-aq.com/v1/devices/MOD-X-0095...
3,None,"{'lat': None, 'lon': None}","{'wx_u': 0.0, 'wx_v': 0.0, 'wx_wd': 0.0, 'wx_w...","{'gas': {'co': None, 'no': None, 'no2': None, ...",None,None,None,None,None,None,2781105,MOD-X-00959,2025-06-16T14:39:45,2025-06-16T14:39:45,https://api.quant-aq.com/v1/devices/MOD-X-0095...
4,None,"{'lat': None, 'lon': None}","{'wx_u': 0.0, 'wx_v': 0.0, 'wx_wd': 0.0, 'wx_w...","{'gas': {'co': None, 'no': None, 'no2': None, ...",None,None,None,None,None,None,2781104,MOD-X-00959,2025-06-16T14:38:45,2025-06-16T14:38:45,https://api.quant-aq.com/v1/devices/MOD-X-0095...
5,None,"{'lat': None, 'lon': None}","{'wx_u': 0.0, 'wx_v': 0.0, 'wx_wd': 0.0, 'wx_w...","{'gas': {'co': None, 'no': None, 'no2': None, ...",None,None,None,None,None,None,2781102,MOD-X-00959,2025-06-16T14:37:45,2025-06-16T14:37:45,https://api.quant-aq.com/v1/devices/MOD-X-0095...
6,None,"{'lat': None, 'lon': None}","{'wx_u': 0.0, 'wx_v': 0.0, 'wx_wd': 0.0, 'wx_w...","{'gas': {'co': None, 'no': None, 'no2': None, ...",None,None,None,None,None,None,2781103,MOD-X-00959,2025-06-16T14:36:45,2025-06-16T14:36:45,https://api.quant-aq.com/v1/devices/MOD-X-0095...
7,None,"{'lat': None, 'lon': None}","{'wx_u': 0.0, 'wx_v': 0.0, 'wx_wd': 0.0, 'wx_w...","{'gas': {'co': None, 'no': None, 'no2': None, ...",None,None,None,None,None,None,2781101,MOD-X-00959,2025-06-16T14:35:45,2025-06-16T14:35:45,https://api.quant-aq.com/v1/devices/MOD-X-0095...
8,None,"{'lat': None, 'lon': None}","{'wx_u': 0.0, 'wx_v': 0.0, 'wx_wd': 0.0, 'wx_w...","{'gas': {'co': None, 'no': None, 'no2': None, ...",None,None,None,None,None,None,2780889,MOD-X-00959,2025-06-16T14:34:45,2025-06-16T14:34:45,https://api.quant-aq.com/v1/devices/MOD-X-0095...
9,None,"{'lat': None, 'lon': None}","{'wx_u': 0.0, 'wx_v': 0.0, 'wx_wd': 0.0, 'wx_w...","{'gas': {'co': None, 'no': None, 'no2': None, ...",None,None,None,None,None,None,2780890,MOD-X-00959,2025-06-16T14:33:45,2025-06-16T14:33:45,https://api.quant-aq.com/v1/devices/MOD-X-0095...


In [ ]:
dict_data['meta']


{'first_url': 'https://api.quant-aq.com/v1/devices/MOD-X-00959/data/?page=1&per_page=50',
 'last_url': 'https://api.quant-aq.com/v1/devices/MOD-X-00959/data/?page=1917&per_page=50',
 'next_url': 'https://api.quant-aq.com/v1/devices/MOD-X-00959/data/?page=2&per_page=50',
 'page': 1,
 'pages': 1917,
 'per_page': 50,
 'prev_url': None,
 'total': 95838}